# 10 · Final report: test scores, significance, breakdowns, errors

**Goal:** score the chosen configurations on **test** (saved blind since Direction 1), compare them with paired tests, and produce the tables of the written report. No configuration is chosen here: every choice was made on val (B, D1, D2) or on train CV + val (D3). A new run (e.g. a later cross-encoder) only needs one line in `RUNS`.

**Additional experiments** (stage `Extra`, run after the final system was chosen, so they do not change it): DeBERTa ce4 trained for 2 epochs, the bge-reranker-base cross-encoder (`11_bge_reranker`), DeBERTa-v3-large (`12_deberta_large_t4x2`), and score blends: scores standardized within each query, then weighted. Each blend weight is the best of 0.1–0.5 on val (bge 0.2 with DeBERTa-base, used again for the MiniLM blend and the final + bge blend; base 0.3 and bge 0.2 with large; large 0.5 with the final system), so the val numbers of blends are slightly optimistic and test is the fair comparison.

**Sections:** 1 data · 2 main table (val and test, every run) · 3 headline table (mean over seeds) · 4 paired tests · 5 by query type · 6 where each label lands · 7 per-query wins and losses · 8 worst test queries of the final model · 9 save.

**Kaggle settings:** CPU, no internet, a few minutes. Also runs locally from `notebooks/` (inputs read from `../../kaggle_outputs`).

**Inputs** (Add Input):
- Dataset `wm-src`, a version whose `data/results/` has the `d3_*` runs. Required.
- Notebook Output **01_preprocess** (`pairs.parquet`, `products_clean.parquet`). Required.
- Notebook Output **07_oof** (assemble version) or the dataset with `ce_deberta_main_ce4.parquet`: test scores of the 5-fold CE. Required.

**Output** (`/kaggle/working/report/`, locally `data/report/`): `results_all.csv`, `results_headline.csv`, `paired_tests.csv`, `by_source.csv`, `label_ranks.csv`, `query_deltas.csv`, `worst_queries.csv`.

In [1]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

src from D:\Workspace\Web Mining\web-mining-project | {'files_sha1': '62eb710944'}


In [2]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

from src.data import PREPROCESS_HINT, find
from src.evaluation import paired_bootstrap, summarize

ON_KAGGLE = Path('/kaggle/input').exists()
INPUT = '/kaggle/input' if ON_KAGGLE else '../../kaggle_outputs'
OUT = Path('/kaggle/working/report') if ON_KAGGLE else ROOT / 'data' / 'report'
RES = ROOT / 'data' / 'results'
CE5 = 'd3_ce_deberta_main_ce4'          # 5-fold CE: val from its run file, test from the assembled 07 file
FINAL = 'd3_lgbm_multiclass'            # final system (chosen on CV + val in 09)

# (stage, configuration, seed, run name); order = order of the tables
RUNS = [
    ('B0', 'Random order', None, 'b0_random'),
    ('B1', 'BM25 title', None, 'b1_bm25_title'),
    ('B1', 'BM25 main', None, 'b1_bm25_main'),
    ('B2', 'Bi-encoder MiniLM-L6 title (zero-shot)', None, 'b2_minilml6_title'),
    ('B2', 'Bi-encoder MiniLM-L6 main (zero-shot)', None, 'b2_minilml6_main'),
    ('B2', 'Bi-encoder bge-base title (zero-shot)', None, 'b2_bge_title'),
    ('B2', 'Bi-encoder bge-base main (zero-shot)', None, 'b2_bge_main'),
    ('B3', 'CE MiniLM MS MARCO title (zero-shot)', None, 'b3_minilm_title_zeroshot'),
    ('B3', 'CE MiniLM title, MSE', 42, 'b3_minilm_title_mse_s42'),
    ('B3', 'CE MiniLM title, MSE', 43, 'b3_minilm_title_mse_s43'),
    ('D1', 'CE MiniLM title_bc, MSE', 42, 'd1_minilm_title_bc_mse_s42'),
    ('D1', 'CE MiniLM title_bc, MSE', 43, 'd1_minilm_title_bc_mse_s43'),
    ('D1', 'CE MiniLM main, MSE', 42, 'd1_minilm_main_mse_s42'),
    ('D1', 'CE MiniLM main, MSE', 43, 'd1_minilm_main_mse_s43'),
    ('D1', 'CE DeBERTa title, MSE', 42, 'd1_deberta_title_mse_s42'),
    ('D1', 'CE DeBERTa main, MSE', 42, 'd1_deberta_main_mse_s42'),
    ('D1', 'CE DeBERTa main, MSE', 43, 'd1_deberta_main_mse_s43'),
    ('D2', 'CE MiniLM title, ce4', 42, 'd2_minilm_title_ce4_s42'),
    ('D2', 'CE MiniLM title, ce4', 43, 'd2_minilm_title_ce4_s43'),
    ('D2', 'CE MiniLM title, ListNet', 42, 'd2_minilm_title_listnet_s42'),
    ('D2', 'CE MiniLM title, ListNet', 43, 'd2_minilm_title_listnet_s43'),
    ('D2', 'CE DeBERTa main, ce4', 42, 'd2_deberta_main_ce4_s42'),
    ('D3', 'CE DeBERTa main ce4, mean of 5 fold models', 42, CE5),
    ('D3', 'LightGBM lambdarank on CE + features', 42, 'd3_lgbm_lambdarank'),
    ('D3', 'LightGBM multiclass on CE + features', 42, 'd3_lgbm_multiclass'),
    ('Extra', 'CE DeBERTa main, ce4, 2 epochs', 42, 'd2_deberta_main_e2_ce4_s42'),
    ('Extra', 'CE bge-reranker-base main, ce4', 42, 'd1_bgerr_main_ce4_s42'),
    ('Extra', 'Blend: DeBERTa ce4 + 0.2 bge-reranker', 42, 'x_deberta_bgerr'),
    ('Extra', 'Blend: DeBERTa ce4 + 0.2 MiniLM main', 42, 'x_deberta_minilm'),
    ('Extra', 'Blend: final + 0.2 bge-reranker', 42, 'x_final_bgerr'),
    ('Extra', 'CE DeBERTa-v3-large main, ce4', 42, 'd1_debertalarge_main_ce4_s42'),
    ('Extra', 'Blend: large + 0.3 DeBERTa-base', 42, 'x_large_base'),
    ('Extra', 'Blend: large + 0.2 bge-reranker', 42, 'x_large_bgerr'),
    ('Extra', 'Blend: final + 0.5 large', 42, 'x_final_large'),
]

# blends of runs listed above: within-query z-scores, weighted sum (see the introduction)
BLENDS = {
    'x_deberta_bgerr': {'d2_deberta_main_ce4_s42': 0.8, 'd1_bgerr_main_ce4_s42': 0.2},
    'x_deberta_minilm': {'d2_deberta_main_ce4_s42': 0.8, 'd1_minilm_main_mse_s42': 0.2},
    'x_final_bgerr': {FINAL: 0.8, 'd1_bgerr_main_ce4_s42': 0.2},
    'x_large_base': {'d1_debertalarge_main_ce4_s42': 0.7, 'd2_deberta_main_ce4_s42': 0.3},
    'x_large_bgerr': {'d1_debertalarge_main_ce4_s42': 0.8, 'd1_bgerr_main_ce4_s42': 0.2},
    'x_final_large': {FINAL: 0.5, 'd1_debertalarge_main_ce4_s42': 0.5},
}

# (question, run a, run b): paired difference b - a on the same queries
PAIRS = [
    ('B2 bge main vs BM25 main', 'b1_bm25_main', 'b2_bge_main'),
    ('B3 fine-tuning vs zero-shot CE', 'b3_minilm_title_zeroshot', 'b3_minilm_title_mse_s42'),
    ('B3 vs BM25 main', 'b1_bm25_main', 'b3_minilm_title_mse_s42'),
    ('D1 MiniLM: title_bc vs title (s42)', 'b3_minilm_title_mse_s42', 'd1_minilm_title_bc_mse_s42'),
    ('D1 MiniLM: main vs title (s42)', 'b3_minilm_title_mse_s42', 'd1_minilm_main_mse_s42'),
    ('D1 DeBERTa: main vs title', 'd1_deberta_title_mse_s42', 'd1_deberta_main_mse_s42'),
    ('D1 main: DeBERTa vs MiniLM (s42)', 'd1_minilm_main_mse_s42', 'd1_deberta_main_mse_s42'),
    ('D1 main: DeBERTa vs MiniLM (s43)', 'd1_minilm_main_mse_s43', 'd1_deberta_main_mse_s43'),
    ('D1 seed: DeBERTa main s43 vs s42', 'd1_deberta_main_mse_s42', 'd1_deberta_main_mse_s43'),
    ('D2 MiniLM: ce4 vs MSE (s42)', 'b3_minilm_title_mse_s42', 'd2_minilm_title_ce4_s42'),
    ('D2 MiniLM: ce4 vs MSE (s43)', 'b3_minilm_title_mse_s43', 'd2_minilm_title_ce4_s43'),
    ('D2 MiniLM: ListNet vs MSE (s42)', 'b3_minilm_title_mse_s42', 'd2_minilm_title_listnet_s42'),
    ('D2 MiniLM: ListNet vs MSE (s43)', 'b3_minilm_title_mse_s43', 'd2_minilm_title_listnet_s43'),
    ('D2 DeBERTa main: ce4 vs MSE', 'd1_deberta_main_mse_s42', 'd2_deberta_main_ce4_s42'),
    ('D3 mean of 5 fold models vs 1 model on all train', 'd2_deberta_main_ce4_s42', CE5),
    ('D3 LightGBM lambdarank vs CE (5 folds)', CE5, 'd3_lgbm_lambdarank'),
    ('D3 LightGBM multiclass vs CE (5 folds)', CE5, 'd3_lgbm_multiclass'),
    ('D3 multiclass vs lambdarank', 'd3_lgbm_lambdarank', 'd3_lgbm_multiclass'),
    ('Final vs B3', 'b3_minilm_title_mse_s42', FINAL),
    ('Final vs BM25 main', 'b1_bm25_main', FINAL),
    ('Extra: DeBERTa ce4 2 epochs vs 1 epoch', 'd2_deberta_main_ce4_s42', 'd2_deberta_main_e2_ce4_s42'),
    ('Extra: bge-reranker vs DeBERTa (main, ce4)', 'd2_deberta_main_ce4_s42', 'd1_bgerr_main_ce4_s42'),
    ('Extra: bge-reranker vs MiniLM (main)', 'd1_minilm_main_mse_s42', 'd1_bgerr_main_ce4_s42'),
    ('Extra: DeBERTa + bge-reranker vs DeBERTa', 'd2_deberta_main_ce4_s42', 'x_deberta_bgerr'),
    ('Extra: DeBERTa + MiniLM vs DeBERTa', 'd2_deberta_main_ce4_s42', 'x_deberta_minilm'),
    ('Extra: final + bge-reranker vs final', FINAL, 'x_final_bgerr'),
    ('Extra: DeBERTa large vs base (main, ce4)', 'd2_deberta_main_ce4_s42', 'd1_debertalarge_main_ce4_s42'),
    ('Extra: DeBERTa large vs final', FINAL, 'd1_debertalarge_main_ce4_s42'),
    ('Extra: large + base vs large', 'd1_debertalarge_main_ce4_s42', 'x_large_base'),
    ('Extra: large + bge-reranker vs large', 'd1_debertalarge_main_ce4_s42', 'x_large_bgerr'),
    ('Extra: final + large vs final', FINAL, 'x_final_large'),
]
BREAKDOWN = ['b1_bm25_main', 'b2_bge_main', 'b3_minilm_title_mse_s42', 'd1_deberta_main_mse_s42', CE5, FINAL]
NAME = {r: (f'{c} (s{s})' if s and sum(x[1] == c for x in RUNS) > 1 else c) for _, c, s, r in RUNS}

## 1. Data and scores

Labels come from `pairs.parquet`. Every run's scores are aligned to the pairs by `example_id` and must cover every pair of the split.

In [3]:
pairs = pd.read_parquet(find('pairs.parquet', PREPROCESS_HINT, INPUT),
                        columns=['example_id', 'query_id', 'query', 'product_id', 'esci_label', 'source', 'split'])
parts = {s: pairs[pairs['split'] == s].sort_values(['query_id', 'example_id']).reset_index(drop=True) for s in ['val', 'test']}
ce5_test = pd.read_parquet(find('ce_deberta_main_ce4.parquet', 'Add Input -> 07_oof (assemble) output', INPUT))
ce5_test = ce5_test[ce5_test['split'] == 'test'][['example_id', 'score']]


def scores(run, split):
    f = RES / 'scores' / f'{run}_{split}.parquet'
    if f.exists():
        sc = pd.read_parquet(f, columns=['example_id', 'score'])
    elif run == CE5 and split == 'test':
        sc = ce5_test
    else:
        raise FileNotFoundError(f'{f.name}: rebuild wm-src with the latest data/results')
    s = parts[split][['example_id']].merge(sc, on='example_id', how='left')['score']
    assert s.notna().all(), f'{run} {split}: scores do not cover every pair'
    return s.to_numpy()


def zscore(split, s):
    """Scores standardized within each query (0 for one-item or constant lists)."""
    s = pd.Series(s)
    g = s.groupby(parts[split]['query_id'].to_numpy())
    return ((s - g.transform('mean')) / g.transform('std').replace(0, np.nan)).fillna(0).to_numpy()


metrics, per_query, score_cache = {}, {}, {}
for _, _, _, run in RUNS:
    for split in ['val', 'test']:
        if run in BLENDS:
            score_cache[run, split] = sum(w * zscore(split, score_cache[r, split]) for r, w in BLENDS[run].items())
        else:
            score_cache[run, split] = scores(run, split)
        metrics[run, split], per_query[run, split] = summarize(parts[split], score_cache[run, split])
print(len(RUNS), 'runs scored on val and test |', {s: f"{p['query_id'].nunique():,} queries" for s, p in parts.items()})

34 runs scored on val and test | {'val': '2,089 queries', 'test': '8,956 queries'}


## 2. Every run (val and test)

nDCG with official gains (E 1, S 0.1, C 0.01, I 0), ties broken at random (3 seeds), 95% bootstrap CI over queries. `ndcg_amazon` uses the gains of Amazon's script (S and C swapped), for comparison with the paper only.

In [4]:
rows = []
for stage, config, seed, run in RUNS:
    v, t = metrics[run, 'val'], metrics[run, 'test']
    rows.append({'stage': stage, 'configuration': config, 'seed': seed, 'run': run,
                 'val_ndcg': v['ndcg'], 'test_ndcg': t['ndcg'], 'test_ci95': t['ndcg_ci95'],
                 'test_ndcg_amazon': t['ndcg_amazon'], 'test_ndcg10': t['ndcg10'], 'test_mrr_e': t['mrr_e'],
                 'test_e_over_s': t['e_over_s']})
results = pd.DataFrame(rows)
pd.set_option('display.width', 250, 'display.max_columns', 30, 'display.max_colwidth', 60)
display(results.drop(columns='run'))

,stage,configuration,seed,val_ndcg,test_ndcg,test_ci95,test_ndcg_amazon,test_ndcg10,test_mrr_e,test_e_over_s
0,B0,Random order,NaN,0.7422,0.7459,"[0.7429, 0.7491]",0.7133,0.5763,0.6087,0.5000
1,B1,BM25 title,NaN,0.8162,0.8209,"[0.8179, 0.824]",0.7996,0.6954,0.7579,0.6103
2,B1,BM25 main,NaN,0.8227,0.8244,"[0.8209, 0.8275]",0.8046,0.7012,0.7595,0.6190
3,B2,Bi-encoder MiniLM-L6 title (zero-shot),NaN,0.8202,0.8281,"[0.8248, 0.8311]",0.8051,0.7091,0.7672,0.6062
4,B2,Bi-encoder MiniLM-L6 main (zero-shot),NaN,0.8264,0.8306,"[0.8276, 0.8339]",0.8081,0.7137,0.7702,0.6136
5,B2,Bi-encoder bge-base title (zero-shot),NaN,0.8418,0.8450,"[0.842, 0.848]",0.8243,0.7366,0.7981,0.6397
6,B2,Bi-encoder bge-base main (zero-shot),NaN,0.8429,0.8471,"[0.844, 0.85]",0.8274,0.7401,0.8023,0.6433
7,B3,CE MiniLM MS MARCO title (zero-shot),NaN,0.8330,0.8340,"[0.8311, 0.8369]",0.8132,0.7163,0.7895,0.6248
8,B3,"CE MiniLM title, MSE",42.0,0.8566,0.8549,"[0.8518, 0.8578]",0.8358,0.7539,0.8174,0.6760
9,B3,"CE MiniLM title, MSE",43.0,0.8578,0.8555,"[0.8523, 0.8586]",0.8367,0.7546,0.8176,0.6786


## 3. Headline table (mean over seeds)

One row per configuration; configurations run with two seeds show the mean and the seed range on test.

In [5]:
headline = (results.groupby(['stage', 'configuration'], sort=False)
            .agg(seeds=('seed', lambda s: ', '.join(str(int(x)) for x in s.dropna()) or '-'),
                 val_ndcg=('val_ndcg', 'mean'), test_ndcg=('test_ndcg', 'mean'),
                 test_min=('test_ndcg', 'min'), test_max=('test_ndcg', 'max'),
                 test_ndcg10=('test_ndcg10', 'mean'), test_mrr_e=('test_mrr_e', 'mean'),
                 test_e_over_s=('test_e_over_s', 'mean'))
            .round(4).reset_index())
display(headline)

,stage,configuration,seeds,val_ndcg,test_ndcg,test_min,test_max,test_ndcg10,test_mrr_e,test_e_over_s
0,B0,Random order,-,0.7422,0.7459,0.7459,0.7459,0.5763,0.6087,0.5000
1,B1,BM25 title,-,0.8162,0.8209,0.8209,0.8209,0.6954,0.7579,0.6103
2,B1,BM25 main,-,0.8227,0.8244,0.8244,0.8244,0.7012,0.7595,0.6190
3,B2,Bi-encoder MiniLM-L6 title (zero-shot),-,0.8202,0.8281,0.8281,0.8281,0.7091,0.7672,0.6062
4,B2,Bi-encoder MiniLM-L6 main (zero-shot),-,0.8264,0.8306,0.8306,0.8306,0.7137,0.7702,0.6136
5,B2,Bi-encoder bge-base title (zero-shot),-,0.8418,0.8450,0.8450,0.8450,0.7366,0.7981,0.6397
6,B2,Bi-encoder bge-base main (zero-shot),-,0.8429,0.8471,0.8471,0.8471,0.7401,0.8023,0.6433
7,B3,CE MiniLM MS MARCO title (zero-shot),-,0.8330,0.8340,0.8340,0.8340,0.7163,0.7895,0.6248
8,B3,"CE MiniLM title, MSE","42, 43",0.8572,0.8552,0.8549,0.8555,0.7543,0.8175,0.6773
9,D1,"CE MiniLM title_bc, MSE","42, 43",0.8592,0.8584,0.8583,0.8586,0.7594,0.8232,0.6823


## 4. Paired tests

Difference b − a of per-query nDCG on the same queries: mean, 95% paired bootstrap CI, and the Wilcoxon signed-rank p-value (queries with equal nDCG dropped). Val is shown next to test to check that what was seen on val holds on test.

In [6]:
rows = []
for question, a, b in PAIRS:
    r = {'question': question, 'a': NAME[a], 'b': NAME[b]}
    for split in ['val', 'test']:
        pa, pb = per_query[a, split], per_query[b, split]
        p = paired_bootstrap(pa, pb)
        d = (pb.set_index('query_id')['ndcg'] - pa.set_index('query_id')['ndcg']).dropna()
        r[f'{split}_diff'], r[f'{split}_ci95'] = p['diff'], p['ci95']
        if split == 'test':
            r['test_wilcoxon_p'] = float(wilcoxon(d[d != 0]).pvalue) if (d != 0).sum() > 10 else np.nan
            r['test_better'], r['test_worse'] = int((d > 1e-9).sum()), int((d < -1e-9).sum())
    rows.append(r)
paired = pd.DataFrame(rows)
paired['test_wilcoxon_p'] = paired['test_wilcoxon_p'].map(lambda p: f'{p:.1e}')
display(paired)

,question,a,b,val_diff,val_ci95,test_diff,test_ci95,test_wilcoxon_p,test_better,test_worse
0,B2 bge main vs BM25 main,BM25 main,Bi-encoder bge-base main (zero-shot),0.0202,"[0.0151, 0.0253]",0.0227,"[0.0202, 0.0252]",4.0e-77,5108,3657
1,B3 fine-tuning vs zero-shot CE,CE MiniLM MS MARCO title (zero-shot),"CE MiniLM title, MSE (s42)",0.0235,"[0.0197, 0.0277]",0.0209,"[0.0189, 0.0229]",1.6e-131,5537,3167
2,B3 vs BM25 main,BM25 main,"CE MiniLM title, MSE (s42)",0.0338,"[0.0278, 0.0396]",0.0305,"[0.0278, 0.0335]",3.0e-106,5256,3572
3,D1 MiniLM: title_bc vs title (s42),"CE MiniLM title, MSE (s42)","CE MiniLM title_bc, MSE (s42)",0.0023,"[-0.0002, 0.0048]",0.0034,"[0.0023, 0.0046]",1.7e-08,4209,3781
4,D1 MiniLM: main vs title (s42),"CE MiniLM title, MSE (s42)","CE MiniLM main, MSE (s42)",0.0104,"[0.0069, 0.0141]",0.0106,"[0.0088, 0.0122]",1.5e-48,4876,3580
5,D1 DeBERTa: main vs title,"CE DeBERTa title, MSE","CE DeBERTa main, MSE (s42)",0.0077,"[0.0043, 0.011]",0.0102,"[0.0086, 0.0118]",2.0e-48,4752,3466
6,D1 main: DeBERTa vs MiniLM (s42),"CE MiniLM main, MSE (s42)","CE DeBERTa main, MSE (s42)",0.0209,"[0.0163, 0.0253]",0.0220,"[0.0199, 0.0241]",1.4e-135,5456,3117
7,D1 main: DeBERTa vs MiniLM (s43),"CE MiniLM main, MSE (s43)","CE DeBERTa main, MSE (s43)",0.0176,"[0.0129, 0.0222]",0.0196,"[0.0175, 0.0217]",1.2e-112,5361,3191
8,D1 seed: DeBERTa main s43 vs s42,"CE DeBERTa main, MSE (s42)","CE DeBERTa main, MSE (s43)",-0.0001,"[-0.0029, 0.003]",-0.0014,"[-0.0025, -0.0001]",1.2e-03,3763,4058
9,D2 MiniLM: ce4 vs MSE (s42),"CE MiniLM title, MSE (s42)","CE MiniLM title, ce4 (s42)",-0.0006,"[-0.0036, 0.0026]",-0.0005,"[-0.0018, 0.0008]",9.3e-01,4207,4109


## 5. By query type

`source` from the dataset: `other` (most queries), `parse_pattern` (structured queries), `negations` ("without", "not", …), `behavioral`, `nlqec` (natural-language queries, very few). Mean test nDCG per type.

In [7]:
src_of = parts['test'].groupby('query_id')['source'].first()
by_source = pd.DataFrame({NAME[r]: per_query[r, 'test'].set_index('query_id')['ndcg'] for r in BREAKDOWN})
by_source['source'] = src_of.reindex(by_source.index)
by_source = by_source.groupby('source').agg(['mean']).droplevel(1, axis=1).round(4)
by_source.insert(0, 'queries', src_of.value_counts())
display(by_source.sort_values('queries', ascending=False))

,queries,BM25 main,Bi-encoder bge-base main (zero-shot),"CE MiniLM title, MSE (s42)","CE DeBERTa main, MSE (s42)","CE DeBERTa main ce4, mean of 5 fold models",LightGBM multiclass on CE + features
source,,,,,,,
other,7233,0.8283,0.8532,0.8563,0.8915,0.8928,0.8966
parse_pattern,781,0.8488,0.8631,0.8726,0.8903,0.8882,0.8898
negations,689,0.7510,0.7601,0.8212,0.8448,0.8454,0.8476
behavioral,223,0.8516,0.8648,0.8651,0.8870,0.8889,0.8931
nlqec,30,0.7225,0.8380,0.7684,0.8526,0.8607,0.8669


## 6. Where each label lands

Mean position of each label inside its list, as a share of the list (0 = top, 1 = bottom; random ≈ 0.5), on test. Exact items should rise and Irrelevant items sink; the gap between E and S is what nDCG rewards most.

In [8]:
def label_positions(run):
    d = parts['test'][['query_id', 'esci_label']].assign(score=score_cache[run, 'test'])
    r = d.groupby('query_id')['score'].rank(ascending=False, method='average')
    n = d.groupby('query_id')['score'].transform('size')
    return ((r - 1) / (n - 1)).groupby(d['esci_label']).mean()


label_ranks = pd.DataFrame({NAME[r]: label_positions(r) for r in BREAKDOWN}).T[['E', 'S', 'C', 'I']].round(3)
label_ranks['S - E'] = (label_ranks['S'] - label_ranks['E']).round(3)
display(label_ranks)

esci_label,E,S,C,I,S - E
BM25 main,0.443,0.542,0.473,0.570,0.099
Bi-encoder bge-base main (zero-shot),0.423,0.543,0.534,0.602,0.120
"CE MiniLM title, MSE (s42)",0.414,0.554,0.552,0.599,0.140
"CE DeBERTa main, MSE (s42)",0.384,0.568,0.629,0.627,0.184
"CE DeBERTa main ce4, mean of 5 fold models",0.383,0.567,0.633,0.632,0.184
LightGBM multiclass on CE + features,0.381,0.568,0.623,0.638,0.187


## 7. Per-query wins and losses: final system vs B3

How many test queries the final system improves or hurts compared with the B3 reproduction, and the size of the changes.

In [9]:
qa = per_query['b3_minilm_title_mse_s42', 'test'].set_index('query_id')['ndcg']
qb = per_query[FINAL, 'test'].set_index('query_id')['ndcg']
delta = (qb - qa).dropna().rename('delta').to_frame()
delta['b3'], delta['final'] = qa, qb
info = parts['test'].groupby('query_id').agg(query=('query', 'first'), source=('source', 'first'), n=('esci_label', 'size'),
                                              n_E=('esci_label', lambda l: int((l == 'E').sum())),
                                              n_S=('esci_label', lambda l: int((l == 'S').sum())))
delta = delta.join(info)
bins = pd.cut(delta['delta'], [-1, -0.1, -0.01, -1e-9, 1e-9, 0.01, 0.1, 1],
              labels=['< -0.1', '-0.1..-0.01', '-0.01..0', 'same', '0..0.01', '0.01..0.1', '> 0.1'])
display(bins.value_counts().reindex(bins.cat.categories).rename('queries').to_frame().T)
show = ['query', 'source', 'n', 'n_E', 'n_S', 'b3', 'final', 'delta']
print('Largest gains'); display(delta.sort_values('delta', ascending=False)[show].head(8).round(3))
print('Largest losses'); display(delta.sort_values('delta')[show].head(8).round(3))

,< -0.1,-0.1..-0.01,-0.01..0,same,0..0.01,0.01..0.1,> 0.1
queries,493,1554,598,288,962,3470,1591


Largest gains


,query,source,n,n_E,n_S,b3,final,delta
query_id,,,,,,,,
52737,ifuser,other,16,1,0,0.250,1.000,0.750
101401,tempered chocolate kit,other,15,1,0,0.356,1.000,0.644
88704,rouge marvel select,other,16,2,3,0.353,0.985,0.632
51582,how front turn signal klr650,other,16,1,4,0.365,0.980,0.614
5723,I would want a my replacement to be a Camo print jacket....,nlqec,27,1,0,0.387,1.000,0.613
47493,grey pocket book,other,16,1,0,0.387,1.000,0.613
84872,quant revolution,other,16,1,0,0.387,1.000,0.613
90438,sanddollar paver maker,other,16,1,6,0.378,0.985,0.608


Largest losses


,query,source,n,n_E,n_S,b3,final,delta
query_id,,,,,,,,
102156,the looming tower,other,15,1,0,1.000,0.315,-0.685
28565,contact lens,other,16,1,0,0.996,0.460,-0.536
30469,csj,other,16,1,0,1.000,0.500,-0.500
41628,fly argic,other,16,1,0,1.000,0.500,-0.500
105261,triggered donald trump jr,other,16,1,0,1.000,0.500,-0.500
25265,chelsea stadium soccer cards,other,16,2,1,1.000,0.503,-0.497
77404,panacur for cats,other,16,1,8,0.921,0.436,-0.485
12755,ball keeper net,other,16,1,11,0.987,0.544,-0.443


## 8. Worst test queries of the final system

The lowest-nDCG queries that have at least one Exact item, with the full ranked list (label, score, title). Used for the error analysis of the report.

In [10]:
titles = pd.read_parquet(find('products_clean.parquet', PREPROCESS_HINT, INPUT), columns=['product_id', 'title'])
titles = titles.set_index('product_id')['title']
t = parts['test'].assign(score=score_cache[FINAL, 'test'])
pq = per_query[FINAL, 'test'].set_index('query_id')
has_e = t.groupby('query_id')['esci_label'].apply(lambda l: (l == 'E').any())
worst = pq.loc[has_e[has_e].index, 'ndcg'].sort_values().head(6)

rows = []
for qid, nd in worst.items():
    q = t[t['query_id'] == qid].sort_values('score', ascending=False)
    print(f"\n[{q['source'].iloc[0]}] \"{q['query'].iloc[0]}\"  nDCG {nd:.3f}  ({len(q)} candidates)")
    for rank, (_, r) in enumerate(q.iterrows(), 1):
        title = str(titles.get(r['product_id'], ''))[:90]
        if rank <= 8 or r['esci_label'] == 'E':
            print(f"  {rank:2d}. {r['esci_label']}  {r['score']:.3f}  {title}")
        rows.append({'query_id': qid, 'query': r['query'], 'ndcg': round(nd, 4), 'rank': rank,
                     'label': r['esci_label'], 'score': round(float(r['score']), 4), 'title': title})
worst_rows = pd.DataFrame(rows)


[negations] "&#34;tortillas without interesterified soybean oil”"  nDCG 0.198  (39 candidates)


   1. I  0.265  Soymerica Non-GMO Soybeans - 7 Lbs (Newest Crop). Identity Preserved (IP). Great for Soy M
   2. I  0.232  Healthy Harvest Non-GMO Gourmet Soybean Cooking Oil - Healthy Cooking Oil for Cooking, Bak
   3. I  0.224  Laura Soybeans, newest crop and Non-GMO - directly from the Chambers Family Farm in Iowa. 
   4. I  0.209  Snappy Popcorn 1 Gallon Snappy Pure Peanut Oil No Color Added, 128 Fl Oz
   5. I  0.188  Seapoint Farms Sea Salt Dry Roasted Edamame, 4 oz Gluten-Free Snack (12 pack)
   6. I  0.172  Castor Oil USDA Organic Cold-Pressed (16oz) 100% Pure Hexane-Free Castor Oil - Conditionin
   7. I  0.172  Healthy Harvest Canola Oil - Non-GMO Certified with Antioxidants and Omega-3s {One Gallon 
   8. I  0.171  FLY BY JING Sichuan Chili Crisp 6oz, Deliciously Savory Umami Spicy Tingly Crispy Gourmet 
  32. E  0.085  Nature Made Fish Oil 1000 mg, 250 Softgels Value Size, Fish Oil Omega 3 Supplement For Hea

[other] "silk bodysuit long sleevetops"  nDCG 0.245  (16 candidates

## 9. Save

In [11]:
OUT.mkdir(parents=True, exist_ok=True)
results.to_csv(OUT / 'results_all.csv', index=False)
headline.to_csv(OUT / 'results_headline.csv', index=False)
paired.to_csv(OUT / 'paired_tests.csv', index=False)
by_source.to_csv(OUT / 'by_source.csv')
label_ranks.to_csv(OUT / 'label_ranks.csv')
delta.sort_values('delta').to_csv(OUT / 'query_deltas.csv')
worst_rows.to_csv(OUT / 'worst_queries.csv', index=False)
print('wrote', sorted(p.name for p in OUT.glob('*.csv')), 'to', OUT)

wrote ['by_source.csv', 'label_ranks.csv', 'paired_tests.csv', 'query_deltas.csv', 'results_all.csv', 'results_headline.csv', 'worst_queries.csv'] to D:\Workspace\Web Mining\web-mining-project\data\report
